# Environment Setup

## Pip the necessary library

In [ ]:
dataset_path = "/kaggle/input/datasets/changchoufang/cv-hw1-data" # please to change to your path

## Library import

In [ ]:
# -*- coding: utf-8 -*-
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import timm
from tqdm import tqdm
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import confusion_matrix

## Set the random seed

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(42)

# Pre-process for inferencing

## Inference config setting

In [55]:
#########################################
# =============== setting ===============
#########################################

# datasetpath (please to change to your path)
train_path = os.path.join(dataset_path, "cv_hw1_data/data/train")
val_path   = os.path.join(dataset_path, "cv_hw1_data/data/val")
test_path  = os.path.join(dataset_path, "cv_hw1_data/data/test")

NUM_CLASSES = 100

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# the path of the model with the highest accuracy
BEST_WEIGHT_PATH = os.path.join(dataset_path, "student_model.pth")

## The function for model and dataset

In [ ]:
#########################################
# ========== model build ==================
#########################################

def build_models():
    
    # Student
    model = timm.create_model(
        'seresnextaa101d_32x8d.sw_in12k_ft_in1k_288',
        pretrained=False,
        num_classes=NUM_CLASSES
    )

    # Load the best model weight
    if os.path.exists(BEST_WEIGHT_PATH):
        model.load_state_dict(torch.load(BEST_WEIGHT_PATH))
        print(f"Successfully loaded the best model weights from {BEST_WEIGHT_PATH}")
    else:
        print(f"CRITICAL WARNING: model weights not found at {BEST_WEIGHT_PATH}!")

    model = model.to(DEVICE)
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
        
    return model

In [57]:
#########################################
# ========== Model Size =================
#########################################

def print_model_size(model):
    total = sum(p.numel() for p in model.parameters())
    print(f"Model Params: {total/1e6:.2f}M")

## Build the model and student model using the function above

In [58]:
# Build model and ensure its model size

model = build_models()
print_model_size(model)

Successfully loaded thwe best model weights from /kaggle/input/datasets/changchoufang/cv-hw1-data/student_model.pth
Model Params: 91.74M


In [64]:
# Using the same transformations as the teacher model during training
# to ensure consistent input preprocessing for inference

val_transform = transforms.Compose([
    transforms.Resize(384, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(384),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Ensures correct mapping between image and class index
train_dataset = ImageFolder(train_path, transform=val_transform)

# Inference the test dataset

## The function of Inference

In [65]:
#########################################
# ===== Inference on Test Set ===========
#########################################

def inference(model, path_name):

    # ===== Map prediction index back to original class label =====
    idx_to_class = {v: int(k) for k, v in train_dataset.class_to_idx.items()}

    # ===== Model setup =====
    model.to(DEVICE)
    model.eval()

    # ===== Load test file names =====
    test_files = sorted(os.listdir(test_path))

    # ====== Inference Loop ======
    results = []
    for f in tqdm(test_files, desc="Testing"):

        # ----- Load and preprocess image -----
        img = Image.open(os.path.join(test_path, f)).convert("RGB")
        img = val_transform(img).unsqueeze(0).to(DEVICE)

        # ----- Forward pass (no gradient) -----
        with torch.no_grad():
            pred = model(img).argmax(1).item()

        # ----- Save result (image_id, predicted label) -----
        results.append([os.path.splitext(f)[0], idx_to_class[pred]])

    # ===== Save submission file =====
    pd.DataFrame(results, columns=["image_name", "pred_label"]).to_csv(path_name, index=False)
    print(f"{path_name} saved!")

## Run the inference process

In [62]:
# run the inference
inference(model, "prediction.csv")

Testing: 100%|██████████| 2344/2344 [02:05<00:00, 18.64it/s]

prediction.csv saved!
